# CBBpy EDA
Exploratory loading of all dataset types available from the CBBpy API.

**Requirements:** `pip install cbbpy` — use Python 3.11 (not 3.14; numpy wheels not yet available).

**API coverage in this notebook:**
- Game IDs by date
- Game info / box score / play-by-play (single game)
- All game data for a team's season
- Team schedule
- Teams in a conference
- Conference schedule
- Date-range batch load
- Player profile

> **Note:** `get_games_season()` is documented at the bottom but intentionally not executed — it fetches every game for a full season (~5,000+ games) and takes 30+ minutes. Reserve it for the actual ETL pipeline.

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
from cbbpy import mens_scraper
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

print('cbbpy loaded OK')

In [ ]:
# Constants used throughout — change these to explore different slices
SEASON = 2025           # year the season ends: 2025 = 2024-25 season
SAMPLE_TEAM = 'Duke'
SAMPLE_CONFERENCE = 'ACC'
SAMPLE_DATE = '2025-03-01'   # regular-season Saturday; adjust if no games found
RANGE_START = '2025-01-01'
RANGE_END   = '2025-01-02'   # keep range short for exploration

---
## 1. Game IDs for a date
`get_game_ids(date)` → list of ESPN game IDs played on that date.

In [ ]:
game_ids = mens_scraper.get_game_ids(SAMPLE_DATE)

print(f'Games on {SAMPLE_DATE}: {len(game_ids)}')
print(game_ids[:10])

SAMPLE_GAME_ID = game_ids[0]
print(f'\nUsing game_id={SAMPLE_GAME_ID} for single-game cells below')

---
## 2. Game Info
`get_game_info(game_id)` → one-row DataFrame: teams, score, date, venue, officials, attendance.

In [ ]:
game_info = mens_scraper.get_game_info(SAMPLE_GAME_ID)

print(f'Shape: {game_info.shape}')
print(f'Columns: {list(game_info.columns)}')
game_info

---
## 3. Game Box Score
`get_game_boxscore(game_id)` → player-level stats for both teams: minutes, points, rebounds, assists, shooting splits, +/-, etc.

In [ ]:
boxscore = mens_scraper.get_game_boxscore(SAMPLE_GAME_ID)

print(f'Shape: {boxscore.shape}  ({boxscore.shape[0]} player-rows)')
print(f'Columns: {list(boxscore.columns)}')
boxscore.head(10)

In [ ]:
# Quick look at dtypes and nulls
print('Dtypes:')
print(boxscore.dtypes)
print('\nNull counts:')
print(boxscore.isnull().sum())

---
## 4. Play-by-Play
`get_game_pbp(game_id)` → event-level log: event type, team, player, shot zone, clock, score.

In [ ]:
pbp = mens_scraper.get_game_pbp(SAMPLE_GAME_ID)

print(f'Shape: {pbp.shape}  ({pbp.shape[0]} events)')
print(f'Columns: {list(pbp.columns)}')
pbp.head(15)

In [ ]:
# Event type distribution — useful for understanding what gets logged
if 'play_type' in pbp.columns:
    print(pbp['play_type'].value_counts())
elif 'event_type' in pbp.columns:
    print(pbp['event_type'].value_counts())
else:
    # Print whichever column looks like a type field
    type_cols = [c for c in pbp.columns if 'type' in c.lower() or 'play' in c.lower()]
    print(f'Candidate type columns: {type_cols}')
    if type_cols:
        print(pbp[type_cols[0]].value_counts())

---
## 5. All Game Data at Once
`get_game(game_id)` → convenience wrapper returning `(info, boxscore, pbp)` in a single call.

In [ ]:
g_info, g_box, g_pbp = mens_scraper.get_game(SAMPLE_GAME_ID)

print(f'Info:     {g_info.shape}')
print(f'Boxscore: {g_box.shape}')
print(f'PBP:      {g_pbp.shape}')

---
## 6. Team Schedule
`get_team_schedule(team, season)` → game-level schedule with results for a given team and season.

In [ ]:
team_schedule = mens_scraper.get_team_schedule(SAMPLE_TEAM, season=SEASON)

print(f'Shape: {team_schedule.shape}')
print(f'Columns: {list(team_schedule.columns)}')
team_schedule.head(10)

---
## 7. All Games for a Team's Season
`get_games_team(team, season)` → full info + box + PBP for every game a team played.

> This is the primary unit for player-level feature engineering per team.

In [ ]:
team_info, team_box, team_pbp = mens_scraper.get_games_team(SAMPLE_TEAM, season=SEASON)

print(f'Info shape:     {team_info.shape}  (one row per game)')
print(f'Boxscore shape: {team_box.shape}  (one row per player per game)')
print(f'PBP shape:      {team_pbp.shape}  (one row per event)')

In [ ]:
team_box.head(10)

In [ ]:
team_pbp.head(10)

---
## 8. Teams in a Conference
`get_teams_from_conference(conference, season)` → list of team name strings.

In [ ]:
conf_teams = mens_scraper.get_teams_from_conference(SAMPLE_CONFERENCE, season=SEASON)

print(f'Teams in {SAMPLE_CONFERENCE} ({SEASON}): {len(conf_teams)}')
print(conf_teams)

---
## 9. Conference Schedule
`get_conference_schedule(conference, season)` → schedule of all conference games (not just intra-conference).

In [ ]:
conf_schedule = mens_scraper.get_conference_schedule(SAMPLE_CONFERENCE, season=SEASON)

print(f'Shape: {conf_schedule.shape}')
print(f'Columns: {list(conf_schedule.columns)}')
conf_schedule.head(10)

---
## 10. Date-Range Batch Load
`get_games_range(start, end)` → all games across all teams in a date window.

> This is the daily ingestion workhorse. Keep the range short here; use 1-day increments in the Airflow ETL.

In [ ]:
range_info, range_box, range_pbp = mens_scraper.get_games_range(RANGE_START, RANGE_END)

print(f'Date range: {RANGE_START} → {RANGE_END}')
print(f'Info:       {range_info.shape}')
print(f'Boxscore:   {range_box.shape}')
print(f'PBP:        {range_pbp.shape}')

In [ ]:
range_info.head()

In [ ]:
range_box.head()

---
## 11. Player Profile
`get_player_info(player_id)` → biographical data: name, position, height, weight, hometown, class year.

Player IDs come from the `player_id` column in box score data.

In [ ]:
# Pull a player_id from the box score loaded above
player_id_col = 'player_id' if 'player_id' in boxscore.columns else boxscore.columns[0]
sample_player_id = boxscore[player_id_col].dropna().iloc[0]
print(f'Sample player_id: {sample_player_id}')

player_info = mens_scraper.get_player_info(sample_player_id)
print(f'Shape: {player_info.shape}')
print(f'Columns: {list(player_info.columns)}')
player_info

---
## 12. Conference Games Batch
`get_games_conference(conference, season)` → all info + box + PBP for every game any team in the conference played.

> Heavier than team-level — loads ~700-1000 games for a major conference. Allow several minutes.

In [ ]:
# conf_info, conf_box, conf_pbp = mens_scraper.get_games_conference(SAMPLE_CONFERENCE, season=SEASON)
# print(f'Info:     {conf_info.shape}')
# print(f'Boxscore: {conf_box.shape}')
# print(f'PBP:      {conf_pbp.shape}')
print('Uncomment to run — takes several minutes for a full conference season.')

---
## 13. Full Season Load (ETL reference — do not run interactively)
`get_games_season(season)` → every D-I game for the season. This is what the Airflow ETL pipeline calls on the historical backfill.

> ~5,000 games × 3 datasets = very long runtime (30+ min). Do not run in this notebook.

In [ ]:
# DO NOT RUN HERE — use the Airflow DAG for full-season loads
# season_info, season_box, season_pbp = mens_scraper.get_games_season(season=SEASON)
print('Reserved for ETL pipeline (airflow/workers/scrape_cbbpy_worker.py).')

---
## Appendix: Column Reference

Run the cells below after executing the data loads above to get a quick column reference for each dataset type.

In [ ]:
datasets = {
    'game_info':  game_info,
    'boxscore':   boxscore,
    'pbp':        pbp,
    'team_schedule': team_schedule,
    'range_info': range_info,
    'range_box':  range_box,
    'player_info': player_info,
}

for name, df in datasets.items():
    print(f'\n── {name} {df.shape} ──')
    for col in df.columns:
        print(f'   {col:35s}  {str(df[col].dtype):10s}  nulls={df[col].isnull().sum()}')